# Bonus Activity - Unsloth GRPO Training on Open R1 Math Raw

### 1. Insall Unsloth for Collab

In [1]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm==0.7.3
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    !pip install --no-deps unsloth vllm==0.7.3

In [2]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft "trl==0.15.2" triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

In [ ]:
import re
import json
import torch
from google.colab import files
import numpy as np
from transformers import AutoModel, AutoTokenizer
from sklearn.metrics.pairwise import cosine_similarity
import wandb
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel
from trl import GRPOConfig, GRPOTrainer
from vllm import SamplingParams
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Initialize wandb
wandb.login()

In [ ]:
max_seq_length = 2048 # Can increase for longer reasoning traces
lora_rank = 16

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "meta-llama/Llama-3.2-3B-Instruct",
    max_seq_length = max_seq_length,
    load_in_4bit = False, # False for LoRA 16bit
    fast_inference = True, # Enable vLLM fast inference
    max_lora_rank = lora_rank,
    gpu_memory_utilization = 0.7, # Reduce if out of memory
)

model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ], # Remove QKVO if out of memory
    lora_alpha = lora_rank,
    use_gradient_checkpointing = "unsloth", # Enable long context finetuning
    random_state = 3407,
)

### Helper functions to extract information and formatting system prompt

In [ ]:
SYSTEM_PROMPT = """
Solve the following mathematical problem. Respond in the following format:
<reasoning>
Detail your step-by-step reasoning here...
</reasoning>
<answer>
Your final answer here...
</answer>
"""

# Functions to extract information from responses
def extract_xml_answer(text: str) -> str:
    """Extract text between <answer> and </answer> tags."""
    if "<answer>" not in text or "</answer>" not in text:
        return ""
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

def extract_xml_reasoning(text: str) -> str:
    """Extract text between <reasoning> and </reasoning> tags."""
    if "<reasoning>" not in text or "</reasoning>" not in text:
        return ""
    reasoning = text.split("<reasoning>")[-1]
    reasoning = reasoning.split("</reasoning>")[0]
    return reasoning.strip()

def clean_math_text(text: str) -> str:
    """Clean mathematical text by normalizing spacing and preserving LaTeX."""
    if not text:
        return ""
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text)
    # Preserve LaTeX delimiters
    text = re.sub(r'(\$+)(.*?)(\$+)', lambda m: m.group(1) + m.group(2).replace(' ', '') + m.group(3), text)
    return text.strip()

def extract_answer_from_solution(solution: str) -> str:
    """
    Extract the answer part from a reference solution.
    Many solutions have the answer in the last paragraph, last line,
    or after "Therefore" or similar phrases.
    """
    if not solution:
        return ""
        
    # First try to find explicit answer markers
    answer_markers = [
        "\\boxed{", "answer is ", "answer:", "final answer:", 
        "thus, ", "therefore, ", "hence, "
    ]
    
    solution_lower = solution.lower()
    for marker in answer_markers:
        if marker in solution_lower:
            answer_part = solution[solution_lower.find(marker):]
            return clean_math_text(answer_part)
    
    # If no markers found, try to get the last paragraph or line
    solution_lines = solution.strip().split("\n")
    if len(solution_lines) > 1:
        # Get the last non-empty line
        for line in reversed(solution_lines):
            if line.strip():
                return clean_math_text(line)
    
    # Fallback to the whole solution
    return clean_math_text(solution)


### Mathematical embedding model to check similarity between generated answers and multiple valid answers of the 

We will not only check the structure and length of the response, we will also ensure that the generated content is consistent and mathematically valid by performing cosine similarity using a specialized mathematical embedding model (MathBERT)

In [ ]:
# Class for mathematical embedding model
class MathEmbeddingModel:
    def __init__(self, model_name="tbs17/MathBERT", device=None):
        self.model_name = model_name
        self.device = device if device else ('cuda' if torch.cuda.is_available() else 'cpu')
        print(f"Loading {model_name} on {self.device}...")
        
        # Load tokenizer and model
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval() 
        
        # Maximum sequence length
        self.max_length = self.model.config.max_position_embeddings if hasattr(self.model.config, 'max_position_embeddings') else 512
        
        print(f"Model loaded successfully with max sequence length: {self.max_length}")
    
    def get_embedding(self, text):
        if not text:
            # Return zero vector if text is empty
            hidden_size = self.model.config.hidden_size
            return np.zeros(hidden_size)
            
        # Clean and prepare text
        cleaned_text = clean_math_text(text)
        
        # Tokenize text
        inputs = self.tokenizer(
            cleaned_text,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_length
        ).to(self.device)
        
        # Generate embeddings
        with torch.no_grad():
            outputs = self.model(**inputs)
            
            # Get embeddings from the last hidden state
            last_hidden_state = outputs.last_hidden_state
            
            # Mean pooling of all tokens
            attention_mask = inputs.attention_mask.unsqueeze(-1)
            embedding = torch.sum(last_hidden_state * attention_mask, 1) / torch.sum(attention_mask, 1)
            embedding = embedding.cpu().numpy()
        
        return embedding[0]
    
     
    def compute_embedding_similarity(self, embedding1, embedding2):
        if embedding1 is None or embedding2 is None:
            return 0.0
            
        # Compute cosine similarity
        similarity = cosine_similarity([embedding1], [embedding2])[0][0]
        
        return similarity

# Instantiate the mathematical embedding model
math_embedding_model = None

def initialize_math_embedding_model(model_name="tbs17/MathBERT"):
    """Initialize the mathematical embedding model."""
    global math_embedding_model
    math_embedding_model = MathEmbeddingModel(model_name)
    return math_embedding_model

# Load and prepare the dataset with train/test split
def get_open_r1_math_data(split="train", test_size=0.1, random_seed=42):
    # Load the data
    data = load_dataset("open-r1/OpenR1-Math-Raw")[split]
    
    # Filter to keep only valid problems with valid solutions
    data = data.filter(lambda x: x["problem_is_valid"] == "Yes" and x["solution_is_valid"] == "Yes")
    
    # Transform the dataset
    data = data.map(lambda x: {
        "prompt": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": x["problem"]}
        ],
        "solution": x["solution"],
        "problem_type": x["problem_type"],
        "answer_type": x["answer"],
        # Extract generations and correctness if they exist
        "generations": x.get("generations", []),
        "correctness": x.get("correctness", {})
    })
    
    # Split the dataset into train and test
    data = data.shuffle(seed=random_seed)
    test_size = int(len(data) * test_size)
    train_size = len(data) - test_size
    
    train_dataset = data.select(range(train_size))
    test_dataset = data.select(range(train_size, len(data)))
    
    print(f"Dataset loaded: {len(train_dataset)} training examples, {len(test_dataset)} test examples")
    
    return train_dataset, test_dataset

def process_dataset_with_embeddings(dataset):
    """
    Process the dataset to extract embeddings for both generated answers and reference solution answers.
    """
    global math_embedding_model
    
    # Initialize math embedding model if not already done
    if math_embedding_model is None:
        initialize_math_embedding_model()
    
    def extract_embeddings(example):
        # Process generations
        answer_embeddings = []
        answers = []
        
        if "generations" in example and len(example["generations"]) > 0:
            for gen in example["generations"]:
                # Extract answer from generation
                ans = extract_xml_answer(gen)
                if ans:
                    answers.append(ans)
                    try:
                        # Generate embedding for this answer
                        embedding = math_embedding_model.get_embedding(ans)
                        answer_embeddings.append(embedding)
                    except Exception as e:
                        print(f"Error generating embedding: {e}")
        
        # Calculate average embedding if we have any
        avg_embedding = None
        if len(answer_embeddings) > 0:
            avg_embedding = np.mean(answer_embeddings, axis=0)
        
        # Process reference solution
        reference_answer = ""
        reference_answer_embedding = None
        
        if "solution" in example and example["solution"]:
            # Extract answer part from the solution
            reference_answer = extract_answer_from_solution(example["solution"])
            
            if reference_answer:
                try:
                    # Generate embedding for the reference answer
                    reference_answer_embedding = math_embedding_model.get_embedding(reference_answer)
                except Exception as e:
                    print(f"Error generating reference embedding: {e}")
        
        return {
            "generated_answers": answers,
            "avg_answer_embedding": avg_embedding,
            "reference_answer": reference_answer,
            "reference_answer_embedding": reference_answer_embedding
        }
    
    # Apply processing
    processed_dataset = dataset.map(extract_embeddings)
    
    return processed_dataset

### Reward functions

We define several reward functions:
- for compliance with the expected format
- for the use of LaTeX, a language used for formatting formulas
- for generating mathematically correct content that closely matches the generated answers
- for generating content that is mathematically close to the expected answer

In [ ]:
def format_reward_func(completions, **kwargs) -> list[float]:
    """Reward responses that follow the requested XML format."""
    pattern = r"<reasoning>[\s\S]*?</reasoning>\s*<answer>[\s\S]*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [bool(re.search(pattern, r, re.DOTALL)) for r in responses]
    
    # Calculate rewards: 1.0 for matching format, 0.0 otherwise
    rewards = [1.0 if match else 0.0 for match in matches]
    
    return rewards

def latex_reward_func(completions, **kwargs) -> list[float]:
    """Reward correct use of LaTeX in responses."""
    responses = [completion[0]["content"] for completion in completions]
    rewards = []
    
    for response in responses:
        # Count well-formed LaTeX formulas
        valid_latex = len(re.findall(r'\$\$[\s\S]*?\$\$|\$[\s\S]*?\$', response))
        
        # Check balance of LaTeX delimiters
        dollar_balance = response.count('$') % 2 == 0
        
        # Check for common LaTeX mathematical constructs
        common_constructs = ['\\frac', '\\sum', '\\int', '\\prod', '\\exists', '\\forall', 
                            '\\mathbb', '\\sin', '\\cos', '\\tan', '\\sqrt']
        has_common_constructs = any(construct in response for construct in common_constructs)
        
        # Reward for valid LaTeX usage (max 0.5)
        reward = min(valid_latex * 0.1, 0.5)
        
        # Reward for balanced delimiters
        reward += 0.3 if dollar_balance else 0.0
        
        # Reward for using common mathematical constructs
        reward += 0.2 if has_common_constructs else 0.0
        
        rewards.append(reward)
    
    return rewards

def generation_similarity_reward(prompts, completions, avg_answer_embedding, **kwargs) -> list[float]:
    """
    Reward function that compares the embedding of the model's generated answer with the average embedding of previously generated answers.
    """
    global math_embedding_model
    
    # Initialize model if not already done
    if math_embedding_model is None:
        initialize_math_embedding_model()
    
    responses = [completion[0]['content'] for completion in completions]
    extracted_answers = [extract_xml_answer(r) for r in responses]
    
    rewards = []
    
    for i, answer in enumerate(extracted_answers):
        reward = 0.0
        
        # Skip if no answer extracted
        if not answer:
            rewards.append(reward)
            continue
        
        # Get current average embedding
        current_avg_embedding = avg_answer_embedding[i] if isinstance(avg_answer_embedding, list) else avg_answer_embedding
        
        if current_avg_embedding is not None:
            try:
                # Generate embedding for the model's answer
                answer_embedding = math_embedding_model.get_embedding(answer)
                
                # Calculate cosine similarity
                similarity = math_embedding_model.compute_embedding_similarity(answer_embedding, current_avg_embedding)
                
                # Normalize similarity to [0, 1] range
                normalized_similarity = (similarity + 1) / 2
                reward = normalized_similarity
                
            except Exception as e:
                print(f"Error calculating embedding similarity: {e}")
        
        rewards.append(reward)
    
    return rewards

def reference_similarity_reward(prompts, completions, reference_answer_embedding, **kwargs) -> list[float]:
    """
    Reward function that compares the embedding of the model's generated answer with the embedding of the reference solution answer.
    """
    global math_embedding_model
    
    # Initialize model if not already done
    if math_embedding_model is None:
        initialize_math_embedding_model()
    
    responses = [completion[0]['content'] for completion in completions]
    extracted_answers = [extract_xml_answer(r) for r in responses]
    
    rewards = []
    
    for i, answer in enumerate(extracted_answers):
        reward = 0.0
        
        # Skip if no answer extracted
        if not answer:
            rewards.append(reward)
            continue
        
        # Get current reference embedding
        current_ref_embedding = reference_answer_embedding[i] if isinstance(reference_answer_embedding, list) else reference_answer_embedding
        
        if current_ref_embedding is not None:
            try:
                # Generate embedding for the model's answer
                answer_embedding = math_embedding_model.get_embedding(answer)
                
                # Calculate cosine similarity
                similarity = math_embedding_model.compute_embedding_similarity(answer_embedding, current_ref_embedding)
                
                # Normalize similarity to [0, 1] range
                normalized_similarity = (similarity + 1) / 2
                reward = normalized_similarity
                
            except Exception as e:
                print(f"Error calculating reference similarity: {e}")
        
        rewards.append(reward)
    
    return rewards

def apply_rewards(prompts, completions, **kwargs):
    """
    Apply all reward functions and combine their results with weighting.
    """
    format_rewards = format_reward_func(completions)
    latex_rewards = latex_reward_func(completions)
    
    # Get embeddings for similarity checks
    avg_answer_embedding = kwargs.get("avg_answer_embedding", None)
    reference_answer_embedding = kwargs.get("reference_answer_embedding", None)
    
    # Calculate similarity rewards
    generation_rewards = generation_similarity_reward(prompts, completions, avg_answer_embedding)
    reference_rewards = reference_similarity_reward(prompts, completions, reference_answer_embedding)
    
    # Combine rewards with weights - give highest weight to reference similarity
    combined_rewards = []
    
    for i in range(len(completions)):
        # Weight each reward component by importance
        reward = (
            0.5 * reference_rewards[i] +     # 50% for similarity to reference solution
            0.2 * generation_rewards[i] +    # 20% for similarity to generated answers
            0.2 * format_rewards[i] +        # 20% for format
            0.1 * latex_rewards[i]           # 10% for LaTeX usage
        )
        combined_rewards.append(reward)
    
    return combined_rewards


In [ ]:
# Model and training configuration
def setup_and_train():
    """Set up and train the model using GRPO."""
    # Load dataset with train/test split
    train_dataset, test_dataset = get_open_r1_math_data(test_size=0.1)
    
    # Initialize the mathematical embedding model
    initialize_math_embedding_model()
    
    # Process datasets to extract embeddings 
    train_dataset = process_dataset_with_embeddings(train_dataset)
    test_dataset = process_dataset_with_embeddings(test_dataset)
    
    # Save the test dataset for later evaluation
    test_dataset.save_to_disk("/content/drive/MyDrive/test_math_dataset")
    
    # Configure model
    max_seq_length = 2048 
    lora_rank = 32 
    
    # Quantization of the model 
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "meta-llama/Meta-Llama-3.1-8B",
        max_seq_length = max_seq_length,
        load_in_4bit = False, # False for LoRA 16bit
        fast_inference = True, # Enable vLLM fast inference
        max_lora_rank = lora_rank,
        gpu_memory_utilization = 0.8,
    )
    
    # Low rank adapter using unsloth
    model = FastLanguageModel.get_peft_model(
        model,
        r = lora_rank,
        target_modules = [
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj",
        ],
        lora_alpha = lora_rank,
        use_gradient_checkpointing = "unsloth",
        random_state = 3407,
    )
    
    # Calculate maximum prompt length
    max_prompt_length = max(train_dataset.map(
        lambda x: {"tokens": tokenizer.apply_chat_template(x["prompt"], add_generation_prompt=True, tokenize=True)},
        batched=True,
    ).map(lambda x: {"length": len(x["tokens"])})["length"])
    
    max_prompt_length = max_prompt_length + 10  # Extra margin
    
    # Configure training
    training_args = GRPOConfig(
        learning_rate = 5e-6,  # Learning rate
        weight_decay = 0.1,
        warmup_ratio = 0.1,
        lr_scheduler_type = "cosine",
        optim = "adamw_torch_fused",
        logging_steps = 1,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = 4,
        num_generations = 8, 
        max_prompt_length = max_prompt_length,
        max_completion_length = max_seq_length - max_prompt_length,
        # num_train_epochs = 1, # Set to 1 for a full training run
        max_steps = 500,
        save_steps = 50,
        max_grad_norm = 0.1,
        report_to = "wandb",
        output_dir = "outputs_math_reasoning",
    )
    
    # Create and configure GRPO trainer
    trainer = GRPOTrainer(
        model=model,
        tokenizer=tokenizer,
        args=training_args,
        reward_fn=apply_rewards,
        train_dataset=train_dataset,
        data_collator=None,  # Will use default collator
    )
    
    # Start training
    trainer.train()
    
    # Save model after training
    model.save_lora("math_reasoning_grpo_lora")
    
    return model, tokenizer, test_dataset


### Functions to evaluate the model

In [ ]:
# On the test dataset
def evaluate_model(model, tokenizer, test_dataset, num_samples=50):
    """
    Evaluate the trained model on the test dataset.
    """
    print(f"\n=== Evaluating model on {min(num_samples, len(test_dataset))} test examples ===")
    
    # Load math embedding model if not already loaded
    global math_embedding_model
    if math_embedding_model is None:
        initialize_math_embedding_model()
    
    # Take a subset of test samples if needed
    if len(test_dataset) > num_samples:
        eval_dataset = test_dataset.select(range(num_samples))
    else:
        eval_dataset = test_dataset
    
    # Metrics to track
    metrics = {
        "total_samples": len(eval_dataset),
        "format_score": 0.0,
        "generation_similarity_score": 0.0,
        "reference_similarity_score": 0.0,
        "latex_score": 0.0,
        "total_score": 0.0,
        "high_gen_similarity_count": 0,
        "high_ref_similarity_count": 0
    }
    
    # Configure sampling parameters
    sampling_params = SamplingParams(
        temperature=0.7,
        top_p=0.95,
        max_tokens=2048,
    )
    
    # Process each sample
    for i, example in enumerate(eval_dataset):
        print(f"\nEvaluating example {i+1}/{len(eval_dataset)}")
        
        # Format input for the model
        text = tokenizer.apply_chat_template(
            example["prompt"],
            tokenize=False, 
            add_generation_prompt=True
        )
        
        # Generate model response
        try:
            output = model.fast_generate(
                text,
                sampling_params=sampling_params,
                lora_request=model.load_lora("math_reasoning_grpo_lora"),
            )[0].outputs[0].text
            
            # Extract answer
            extracted_answer = extract_xml_answer(output)
            
            # Calculate individual reward components
            format_reward = format_reward_func([[{"content": output}]])[0]
            latex_reward = latex_reward_func([[{"content": output}]])[0]
            
            # Calculate embedding similarities
            generation_similarity = 0.0
            reference_similarity = 0.0
            
            if extracted_answer:
                try:
                    # Generate embedding for the model's answer
                    answer_embedding = math_embedding_model.get_embedding(extracted_answer)
                    
                    # Calculate similarity with average generation embedding
                    if example.get("avg_answer_embedding") is not None:
                        gen_similarity = math_embedding_model.compute_embedding_similarity(
                            answer_embedding, 
                            example["avg_answer_embedding"]
                        )
                        generation_similarity = (gen_similarity + 1) / 2
                        
                        # Count high generation similarity matches
                        if generation_similarity > 0.8:
                            metrics["high_gen_similarity_count"] += 1
                    
                    # Calculate similarity with reference solution embedding
                    if example.get("reference_answer_embedding") is not None:
                        ref_similarity = math_embedding_model.compute_embedding_similarity(
                            answer_embedding, 
                            example["reference_answer_embedding"]
                        )
                        reference_similarity = (ref_similarity + 1) / 2
                        
                        # Count high reference similarity matches
                        if reference_similarity > 0.8:
                            metrics["high_ref_similarity_count"] += 1
                        
                except Exception as e:
                    print(f"Error calculating embedding similarities: {e}")
            
            # Calculate total reward using the same weights as in training
            total_reward = (
                0.5 * reference_similarity +
                0.2 * generation_similarity +
                0.2 * format_reward +
                0.1 * latex_reward
            )
            
            # Update metrics
            metrics["format_score"] += format_reward
            metrics["generation_similarity_score"] += generation_similarity
            metrics["reference_similarity_score"] += reference_similarity
            metrics["latex_score"] += latex_reward
            metrics["total_score"] += total_reward
            
            # Print progress
            print(f"  Score: {total_reward:.3f} (format: {format_reward:.2f}, "
                  f"gen_sim: {generation_similarity:.2f}, "
                  f"ref_sim: {reference_similarity:.2f}, "
                  f"latex: {latex_reward:.2f})")
            
            if extracted_answer:
                print(f"  Model answer: {extracted_answer}")
                print(f"  Reference answer: {example.get('reference_answer', '')}")
            
        except Exception as e:
            print(f"Error evaluating example {i}: {e}")
            metrics["total_samples"] -= 1
    
    # Calculate averages
    if metrics["total_samples"] > 0:
        metrics["format_score"] /= metrics["total_samples"]
        metrics["generation_similarity_score"] /= metrics["total_samples"]
        metrics["reference_similarity_score"] /= metrics["total_samples"]
        metrics["latex_score"] /= metrics["total_samples"]
        metrics["total_score"] /= metrics["total_samples"]
        metrics["high_gen_similarity_rate"] = metrics["high_gen_similarity_count"] / metrics["total_samples"]
        metrics["high_ref_similarity_rate"] = metrics["high_ref_similarity_count"] / metrics["total_samples"]
    
    # Print summary
    print("\n=== Evaluation Results ===")
    print(f"Total samples evaluated: {metrics['total_samples']}")
    print(f"Average format score: {metrics['format_score']:.4f}")
    print(f"Average generation similarity score: {metrics['generation_similarity_score']:.4f}")
    print(f"Average reference similarity score: {metrics['reference_similarity_score']:.4f}")
    print(f"Average LaTeX score: {metrics['latex_score']:.4f}")
    print(f"Average total score: {metrics['total_score']:.4f}")
    print(f"High generation similarity rate: {metrics['high_gen_similarity_rate']:.4f}")
    print(f"High reference similarity rate: {metrics['high_ref_similarity_rate']:.4f}")
    
    return metrics

# On a specific problem
def test_model_with_problem(model, tokenizer, problem_text):
    """
    Test the trained model with a mathematical problem.
    """
    # Prepare input
    text = tokenizer.apply_chat_template([
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem_text},
    ], tokenize=False, add_generation_prompt=True)
    
    # Configure generation parameters
    sampling_params = SamplingParams(
        temperature=0.7,
        top_p=0.95,
        max_tokens=2048,  # More tokens for longer proofs
    )
    
    # Generate response with trained LoRA
    output = model.fast_generate(
        text,
        sampling_params=sampling_params,
        lora_request=model.load_lora("math_reasoning_grpo_lora"),
    )[0].outputs[0].text
    
    return output



### Train the model and save test dataset

In [ ]:
# Train the model and save test dataset
model, tokenizer, test_dataset = setup_and_train()

### Evaluate model with the test dataset

In [ ]:
evaluation_results = evaluate_model(model, tokenizer, test_dataset)
    
# Save evaluation results
with open("evaluation_results.json", "w") as f:
    json.dump(evaluation_results, f, indent=2)


### Test with a specific example

In [ ]:

test_problem = """
Problem. Find all prime numbers p for which there exist positive integers x, y and z such that
x^p + y^p + z^p - x - y - z
is a product of exactly three distinct prime numbers.
"""

result = test_model_with_problem(model, tokenizer, test_problem)
print("\nModel Output for Test Problem:")